In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

df = pd.read_csv("../data_processed/metro_trips_clean.csv")
stations = df[['metro_station', 'line', 'latitude', 'longitude']].drop_duplicates().reset_index(drop=True)
print(stations.shape)
print(stations['line'].value_counts())
print(stations.head())

(53, 4)
line
Red                        26
Green                      18
Red (Route 2020)            6
Red/Green (Interchange)     2
Name: count, dtype: int64
                         metro_station   line   latitude  longitude
0                   ADCB Metro Station    Red  25.244510  55.298183
1  Abu Baker Al Siddique Metro Station  Green  25.270893  55.332755
2               Abu Hail Metro Station  Green  25.275416  55.346450
3     Airport Terminal 1 Metro Station    Red  25.248477  55.352356
4     Airport Terminal 3 Metro Station    Red  25.245027  55.359497


In [2]:
from scipy.spatial.distance import cdist

def order_stations_nn(group_df, start_idx=None):
    coords = group_df[['latitude','longitude']].values
    n = len(coords)
    if start_idx is None:
        # start from the most extreme point (max latitude = northernmost) as a sensible terminus
        start_idx = np.argmax(coords[:,0])
    visited = [start_idx]
    remaining = set(range(n)) - {start_idx}
    while remaining:
        last = coords[visited[-1]]
        remaining_list = list(remaining)
        dists = cdist([last], coords[remaining_list])[0]
        nearest = remaining_list[np.argmin(dists)]
        visited.append(nearest)
        remaining.remove(nearest)
    return group_df.iloc[visited].reset_index(drop=True)

# Order Green line stations (includes the two interchanges as endpoints of the Green portion near BurJuman)
green_stations = stations[stations['line'].isin(['Green','Red/Green (Interchange)'])].copy()
green_ordered = order_stations_nn(green_stations)
print("GREEN LINE ORDER:")
print(green_ordered['metro_station'].tolist())

GREEN LINE ORDER:
['Stadium Metro Station', 'Al Qiyadah Metro Station', 'Abu Hail Metro Station', 'Abu Baker Al Siddique Metro Station', 'Salah Al Din Metro Station', 'Union Metro Station', 'Baniyas Square Metro Station', 'Gold Souq Metro Station', 'Al Ras Metro Station', 'Al Ghubaiba Metro Station', 'Al Fahidi Metro Station', 'BurJuman Metro Station', 'Oud Metha Metro Station', 'Dubai Healthcare City Metro Station', 'Al Jadaf Metro Station', 'Creek Metro Station', 'Al Nahda Metro Station', 'Dubai Airport Free Zone Metro Station', 'Al Qusais Metro Station', 'Etisalat Metro Station']


In [3]:
def order_stations_nn(group_df, start_station_name):
    coords = group_df[['latitude','longitude']].values
    names = group_df['metro_station'].values
    start_idx = np.where(names == start_station_name)[0][0]
    n = len(coords)
    visited = [start_idx]
    remaining = set(range(n)) - {start_idx}
    while remaining:
        last = coords[visited[-1]]
        remaining_list = list(remaining)
        dists = cdist([last], coords[remaining_list])[0]
        nearest = remaining_list[np.argmin(dists)]
        visited.append(nearest)
        remaining.remove(nearest)
    return group_df.iloc[visited].reset_index(drop=True)

green_ordered = order_stations_nn(green_stations, 'Etisalat Metro Station')
print("GREEN LINE ORDER:")
print(green_ordered['metro_station'].tolist())

GREEN LINE ORDER:
['Etisalat Metro Station', 'Al Qusais Metro Station', 'Dubai Airport Free Zone Metro Station', 'Al Nahda Metro Station', 'Stadium Metro Station', 'Al Qiyadah Metro Station', 'Abu Hail Metro Station', 'Abu Baker Al Siddique Metro Station', 'Salah Al Din Metro Station', 'Union Metro Station', 'Baniyas Square Metro Station', 'Gold Souq Metro Station', 'Al Ras Metro Station', 'Al Ghubaiba Metro Station', 'Al Fahidi Metro Station', 'BurJuman Metro Station', 'Oud Metha Metro Station', 'Dubai Healthcare City Metro Station', 'Al Jadaf Metro Station', 'Creek Metro Station']


In [4]:
red_stations = stations[stations['line'].isin(['Red','Red/Green (Interchange)'])].copy()
print(red_stations['metro_station'].tolist())

['ADCB Metro Station', 'Airport Terminal 1 Metro Station', 'Airport Terminal 3 Metro Station', 'Al Jafiliya Metro Station', 'Al Rigga Metro Station', 'BurJuman Metro Station', 'Business Bay Metro Station', 'DAMAC Properties Metro Station', 'DMCC Metro Station', 'Danube Metro Station', 'Deira City Centre Metro Station', 'Dubai Internet City Metro Station', 'Emirates Metro Station', 'Emirates Towers Metro Station', 'Energy Metro Station', 'Financial Centre Metro Station', 'First Abu Dhabi Bank Metro Station', 'GGICO Metro Station', 'Ibn Battuta Metro Station', 'Jabal Ali Metro Station', 'Mall of the Emirates Metro Station', 'Mashreq Metro Station', 'Nakheel Metro Station', 'Noor Bank Metro Station', 'Rashidiya Metro Station', 'UAE Exchange Metro Station', 'Union Metro Station', 'World Trade Centre Metro Station']


In [5]:
red_ordered = order_stations_nn(red_stations, 'Rashidiya Metro Station')
print("RED LINE ORDER:")
print(red_ordered['metro_station'].tolist())

RED LINE ORDER:
['Rashidiya Metro Station', 'Emirates Metro Station', 'Airport Terminal 3 Metro Station', 'Airport Terminal 1 Metro Station', 'GGICO Metro Station', 'Deira City Centre Metro Station', 'Al Rigga Metro Station', 'Union Metro Station', 'BurJuman Metro Station', 'ADCB Metro Station', 'Al Jafiliya Metro Station', 'World Trade Centre Metro Station', 'Emirates Towers Metro Station', 'Financial Centre Metro Station', 'Business Bay Metro Station', 'Noor Bank Metro Station', 'First Abu Dhabi Bank Metro Station', 'Mall of the Emirates Metro Station', 'Mashreq Metro Station', 'Dubai Internet City Metro Station', 'Nakheel Metro Station', 'DAMAC Properties Metro Station', 'DMCC Metro Station', 'Jabal Ali Metro Station', 'Ibn Battuta Metro Station', 'Energy Metro Station', 'Danube Metro Station', 'UAE Exchange Metro Station']


In [6]:
route2020 = stations[stations['line']=='Red (Route 2020)'].copy()
print(route2020['metro_station'].tolist())

# Order this branch starting from Jabal Ali's location (the branch point)
jabal_ali_coord = stations[stations['metro_station']=='Jabal Ali Metro Station'][['latitude','longitude']].values[0]

# Add Jabal Ali as a virtual starting anchor, order, then drop it (since it's already in the trunk)
route2020_with_anchor = pd.concat([
    stations[stations['metro_station']=='Jabal Ali Metro Station'],
    route2020
], ignore_index=True)

route2020_ordered = order_stations_nn(route2020_with_anchor, 'Jabal Ali Metro Station')
print("\nROUTE 2020 BRANCH ORDER (from Jabal Ali):")
print(route2020_ordered['metro_station'].tolist())

['Al Furjan Metro Station', 'Discovery Gardens Metro Station', 'Dubai Investment Park Metro Station', 'EXPO Metro Station', 'Jumeirah Golf Estates Metro Station', 'The Gardens Metro Station']

ROUTE 2020 BRANCH ORDER (from Jabal Ali):
['Jabal Ali Metro Station', 'The Gardens Metro Station', 'Discovery Gardens Metro Station', 'Al Furjan Metro Station', 'Jumeirah Golf Estates Metro Station', 'Dubai Investment Park Metro Station', 'EXPO Metro Station']


In [7]:
from scipy.spatial.distance import euclidean

branch_a = ['Jabal Ali Metro Station', 'The Gardens Metro Station', 'Discovery Gardens Metro Station', 
            'Al Furjan Metro Station', 'Jumeirah Golf Estates Metro Station']
branch_b = ['Jabal Ali Metro Station', 'Dubai Investment Park Metro Station', 'EXPO Metro Station']

# Verify: is EXPO/DIP genuinely closer to Jabal Ali directly than routed via Jumeirah Golf Estates?
coords = stations.set_index('metro_station')[['latitude','longitude']]
d_jabal_dip = euclidean(coords.loc['Jabal Ali Metro Station'], coords.loc['Dubai Investment Park Metro Station'])
d_jge_dip = euclidean(coords.loc['Jumeirah Golf Estates Metro Station'], coords.loc['Dubai Investment Park Metro Station'])
print(f"Distance Jabal Ali -> Dubai Investment Park: {d_jabal_dip:.4f}")
print(f"Distance Jumeirah Golf Estates -> Dubai Investment Park: {d_jge_dip:.4f}")
print("Confirms separate branch from Jabal Ali:" , d_jabal_dip < d_jge_dip)

print("\nBranch A (to Jumeirah Golf Estates):", branch_a)
print("Branch B (to EXPO):", branch_b)

Distance Jabal Ali -> Dubai Investment Park: 0.0598
Distance Jumeirah Golf Estates -> Dubai Investment Park: 0.0147
Confirms separate branch from Jabal Ali: False

Branch A (to Jumeirah Golf Estates): ['Jabal Ali Metro Station', 'The Gardens Metro Station', 'Discovery Gardens Metro Station', 'Al Furjan Metro Station', 'Jumeirah Golf Estates Metro Station']
Branch B (to EXPO): ['Jabal Ali Metro Station', 'Dubai Investment Park Metro Station', 'EXPO Metro Station']


In [8]:
line_order = {
    'Green': green_ordered['metro_station'].tolist(),
    'Red_trunk': red_ordered['metro_station'].tolist(),
    'Red_branch_JGE': branch_a,
    'Red_branch_EXPO': branch_b,
}

import json
with open("../data_processed/line_order.json", "w") as f:
    json.dump(line_order, f, indent=2)

print("Saved line ordering.")
for k,v in line_order.items():
    print(f"\n{k} ({len(v)} stations):", v)

Saved line ordering.

Green (20 stations): ['Etisalat Metro Station', 'Al Qusais Metro Station', 'Dubai Airport Free Zone Metro Station', 'Al Nahda Metro Station', 'Stadium Metro Station', 'Al Qiyadah Metro Station', 'Abu Hail Metro Station', 'Abu Baker Al Siddique Metro Station', 'Salah Al Din Metro Station', 'Union Metro Station', 'Baniyas Square Metro Station', 'Gold Souq Metro Station', 'Al Ras Metro Station', 'Al Ghubaiba Metro Station', 'Al Fahidi Metro Station', 'BurJuman Metro Station', 'Oud Metha Metro Station', 'Dubai Healthcare City Metro Station', 'Al Jadaf Metro Station', 'Creek Metro Station']

Red_trunk (28 stations): ['Rashidiya Metro Station', 'Emirates Metro Station', 'Airport Terminal 3 Metro Station', 'Airport Terminal 1 Metro Station', 'GGICO Metro Station', 'Deira City Centre Metro Station', 'Al Rigga Metro Station', 'Union Metro Station', 'BurJuman Metro Station', 'ADCB Metro Station', 'Al Jafiliya Metro Station', 'World Trade Centre Metro Station', 'Emirates Tow

In [9]:
print(stations[stations['metro_station'].str.contains('Burj Khalifa', na=False)])
print("\nTotal stations with a line assigned:", stations['line'].notna().sum())
print("Stations with NO line assigned:", stations[stations['line'].isna()]['metro_station'].tolist())

                            metro_station line  latitude  longitude
17  Burj Khalifa/Dubai Mall Metro Station  NaN       NaN        NaN

Total stations with a line assigned: 52
Stations with NO line assigned: ['Burj Khalifa/Dubai Mall Metro Station']


In [10]:
df_master_final = pd.read_csv("../data_processed/metro_trips_clean.csv")

# Confirm the bug in the master file
print("Line/coords for Burj Khalifa in master file:")
print(df_master_final[df_master_final['metro_station'].str.contains('Burj Khalifa', na=False)][['metro_station','line','latitude','longitude']].head(3))

Line/coords for Burj Khalifa in master file:
                              metro_station line  latitude  longitude
1605  Burj Khalifa/Dubai Mall Metro Station  NaN       NaN        NaN
1606  Burj Khalifa/Dubai Mall Metro Station  NaN       NaN        NaN
1607  Burj Khalifa/Dubai Mall Metro Station  NaN       NaN        NaN


In [11]:
mask = df_master_final['metro_station'] == 'Burj Khalifa/Dubai Mall Metro Station'
df_master_final.loc[mask, 'line'] = 'Red'
df_master_final.loc[mask, 'latitude'] = 25.2014367
df_master_final.loc[mask, 'longitude'] = 55.2695451

# Verify fix
print(df_master_final[df_master_final['metro_station'].str.contains('Burj Khalifa', na=False)][['metro_station','line','latitude','longitude']].head(3))

df_master_final.to_csv("../data_processed/metro_trips_clean.csv", index=False)
print("\nMaster file re-saved.")

# Regenerate model-ready file from corrected master
cutoff_final = '2023-09-01'
df_model_final = df_master_final[df_master_final['date'] < cutoff_final].copy()
df_model_final.to_csv("../data_processed/metro_trips_model_ready.csv", index=False)
print("Model-ready file regenerated:", df_model_final.shape)

# Reload stations dataframe fresh, confirm all 53 now have line+coords
stations = df_master_final[['metro_station', 'line', 'latitude', 'longitude']].drop_duplicates().reset_index(drop=True)
print("\nStations with line assigned:", stations['line'].notna().sum())
print("Stations with coords assigned:", stations['latitude'].notna().sum())

                              metro_station line   latitude  longitude
1605  Burj Khalifa/Dubai Mall Metro Station  Red  25.201437  55.269545
1606  Burj Khalifa/Dubai Mall Metro Station  Red  25.201437  55.269545
1607  Burj Khalifa/Dubai Mall Metro Station  Red  25.201437  55.269545

Master file re-saved.
Model-ready file regenerated: (3405, 10)

Stations with line assigned: 53
Stations with coords assigned: 53


In [12]:
red_stations = stations[stations['line'].isin(['Red','Red/Green (Interchange)'])].copy()
red_ordered = order_stations_nn(red_stations, 'Rashidiya Metro Station')
print("RED LINE ORDER (with Burj Khalifa/Dubai Mall included):")
print(red_ordered['metro_station'].tolist())

RED LINE ORDER (with Burj Khalifa/Dubai Mall included):
['Rashidiya Metro Station', 'Emirates Metro Station', 'Airport Terminal 3 Metro Station', 'Airport Terminal 1 Metro Station', 'GGICO Metro Station', 'Deira City Centre Metro Station', 'Al Rigga Metro Station', 'Union Metro Station', 'BurJuman Metro Station', 'ADCB Metro Station', 'Al Jafiliya Metro Station', 'World Trade Centre Metro Station', 'Emirates Towers Metro Station', 'Financial Centre Metro Station', 'Burj Khalifa/Dubai Mall Metro Station', 'Business Bay Metro Station', 'Noor Bank Metro Station', 'First Abu Dhabi Bank Metro Station', 'Mall of the Emirates Metro Station', 'Mashreq Metro Station', 'Dubai Internet City Metro Station', 'Nakheel Metro Station', 'DAMAC Properties Metro Station', 'DMCC Metro Station', 'Jabal Ali Metro Station', 'Ibn Battuta Metro Station', 'Energy Metro Station', 'Danube Metro Station', 'UAE Exchange Metro Station']


In [13]:
line_order = {
    'Green': green_ordered['metro_station'].tolist(),
    'Red_trunk': red_ordered['metro_station'].tolist(),
    'Red_branch_JGE': branch_a,
    'Red_branch_EXPO': branch_b,
}

import json
with open("../data_processed/line_order.json", "w") as f:
    json.dump(line_order, f, indent=2)

total_unique = len(set(line_order['Green'] + line_order['Red_trunk'] + line_order['Red_branch_JGE'] + line_order['Red_branch_EXPO']))
print("Total unique stations across all line orderings:", total_unique)

print("Saved final, corrected line ordering.")

Total unique stations across all line orderings: 53
Saved final, corrected line ordering.


In [1]:
import json

# Schematic (non-geographic) station positions, matching RTA's official map style
# Validated earlier via the interactive prototype and cross-checked against the reference map image
schematic_positions = {
    # Green Line
    "Etisalat Metro Station": [640, 60], "Al Qusais Metro Station": [580, 60],
    "Dubai Airport Free Zone Metro Station": [520, 60], "Al Nahda Metro Station": [460, 60],
    "Stadium Metro Station": [400, 60], "Al Qiyadah Metro Station": [340, 60],
    "Abu Hail Metro Station": [340, 98], "Abu Baker Al Siddique Metro Station": [340, 136],
    "Salah Al Din Metro Station": [340, 174], "Union Metro Station": [340, 212],
    "Baniyas Square Metro Station": [340, 250], "Gold Souq Metro Station": [340, 288],
    "Al Ras Metro Station": [340, 326], "Al Ghubaiba Metro Station": [340, 364],
    "Al Fahidi Metro Station": [340, 402], "BurJuman Metro Station": [340, 402],
    "Oud Metha Metro Station": [400, 402], "Dubai Healthcare City Metro Station": [460, 402],
    "Al Jadaf Metro Station": [520, 402], "Creek Metro Station": [580, 402],
    # Red Line top row
    "Rashidiya Metro Station": [760, 212], "Emirates Metro Station": [700, 212],
    "Airport Terminal 3 Metro Station": [640, 212], "Airport Terminal 1 Metro Station": [580, 212],
    "GGICO Metro Station": [520, 212], "Deira City Centre Metro Station": [460, 212],
    "Al Rigga Metro Station": [400, 212],
    # Red Line vertical trunk
    "ADCB Metro Station": [340, 438], "Al Jafiliya Metro Station": [340, 472],
    "World Trade Centre Metro Station": [340, 506], "Emirates Towers Metro Station": [340, 540],
    "Financial Centre Metro Station": [340, 574], "Burj Khalifa/Dubai Mall Metro Station": [340, 608],
    "Business Bay Metro Station": [340, 642], "Noor Bank Metro Station": [340, 676],
    "First Abu Dhabi Bank Metro Station": [340, 710], "Mall of the Emirates Metro Station": [340, 744],
    "Mashreq Metro Station": [340, 778], "Dubai Internet City Metro Station": [340, 812],
    "Nakheel Metro Station": [340, 846], "DAMAC Properties Metro Station": [340, 880],
    "DMCC Metro Station": [340, 914], "Jabal Ali Metro Station": [340, 948],
    # Red Line main trunk continues
    "Ibn Battuta Metro Station": [340, 982], "Energy Metro Station": [340, 1016],
    "Danube Metro Station": [340, 1050], "UAE Exchange Metro Station": [340, 1084],
    # Route 2020: Jumeirah Golf Estates branch
    "The Gardens Metro Station": [400, 948], "Discovery Gardens Metro Station": [460, 948],
    "Al Furjan Metro Station": [520, 948], "Jumeirah Golf Estates Metro Station": [580, 948],
    # Route 2020: EXPO branch
    "Dubai Investment Park Metro Station": [520, 1050], "EXPO Metro Station": [520, 1090],
}

assert len(schematic_positions) == 53, f"Expected 53, got {len(schematic_positions)}"

with open("../data_processed/schematic_positions.json", "w") as f:
    json.dump(schematic_positions, f, indent=2)
print("Saved schematic_positions.json —", len(schematic_positions), "stations")

Saved schematic_positions.json — 53 stations


In [11]:
import json
with open("../data_processed/line_order.json") as f:
    lo = json.load(f)
print(lo['Red_branch_EXPO'])

['Jabal Ali Metro Station', 'Dubai Investment Park Metro Station', 'EXPO Metro Station']
